# Phase 3 — Breeze-ASR-26 深度評測（160 runs）

**對象**：MediaTek-Research/Breeze-ASR-26　·　**資料**：20 難中文 + 20 台語，各配 4 種噪音 = **160 runs**

流程：載入 Breeze（BF16）→ 對每條音檔注入 clean / codec / echo / babble → 30s 分塊 greedy 推論 → OpenCC 繁化算 CER。

## 0. 環境檢查 + 安裝相依

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers==4.46.3 accelerate==1.1.1 librosa==0.10.2 soundfile==0.12.1 opencc-python-reimplemented pandas tqdm

## 1. 掛 Drive + 設路徑

把整個 `KGI_STT_PoC_Bundle/` 上傳到雲端硬碟根目錄。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, pathlib
BUNDLE = pathlib.Path('/content/drive/MyDrive/KGI_STT_PoC_Bundle')

# 共用工具：噪音注入 + CER（與 Phase 1 同源，放在本階段 scripts/）
sys.path.insert(0, str(BUNDLE / '03_phase2_breeze_deep' / 'scripts'))

MANDARIN_CSV   = BUNDLE / '02_phase1_hokkien_batch' / 'data' / 'hard_cases.csv'
HOKKIEN_CSV    = BUNDLE / '02_phase1_hokkien_batch' / 'data' / 'hokkien_cases.csv'
MANDARIN_AUDIO = BUNDLE / '01_streaming_benchmark'  / 'audio'           # d-series
HOKKIEN_AUDIO  = BUNDLE / '02_phase1_hokkien_batch' / 'audio'           # h-series

RESULTS_DIR = BUNDLE / '03_phase2_breeze_deep' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = RESULTS_DIR / 'phase3_breeze_deep_results.csv'

for p in (MANDARIN_CSV, HOKKIEN_CSV, MANDARIN_AUDIO, HOKKIEN_AUDIO):
    assert p.exists(), f'missing: {p}'
print('paths OK →', OUT_CSV)

## 2. 載入 Breeze-ASR-26（BF16）

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

MODEL_ID = 'MediaTek-Research/Breeze-ASR-26'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch_dtype = torch.bfloat16 if device == 'cuda' else torch.float32

print(f'loading {MODEL_ID}  device={device}  dtype={torch_dtype}')
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch_dtype, low_cpu_mem_usage=True
).to(device)
model.eval()

asr = pipeline(
    'automatic-speech-recognition',
    model=model, tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,          # Whisper 30s 手動分塊
    batch_size=1, torch_dtype=torch_dtype, device=device,
)
print('model ready')

## 3. 噪音 + CER 工具（共用 `noise.py` / `metrics.py`）

In [ ]:
import numpy as np, librosa
from noise import add_noise          # clean/codec/echo/babble 注入
from metrics import cer_percent      # OpenCC s2tw 繁化 → 去標點 → Levenshtein

NOISE_TYPES = ['clean', 'codec', 'echo', 'babble']
SNR_DB = 15.0

def load_noisy(wav_path, noise_type):
    """回傳 (float32 audio @16k, sr)；clean 不加噪。"""
    audio_f32, sr = librosa.load(str(wav_path), sr=16000, mono=True)
    if noise_type == 'clean':
        return audio_f32, sr
    audio_i16 = np.clip(audio_f32 * 32767.0, -32768, 32767).astype(np.int16)
    noisy_i16 = add_noise(audio_i16, noise_type, snr_db=SNR_DB)
    return noisy_i16.astype(np.float32) / 32768.0, sr

GEN_KWARGS = dict(task='transcribe', language='zh', num_beams=1, return_timestamps=False)

def transcribe(audio_array, sr):
    out = asr({'array': audio_array, 'sampling_rate': sr}, generate_kwargs=GEN_KWARGS)
    return (out['text'] if isinstance(out, dict) else out).strip()

print('noise types:', NOISE_TYPES, '| SNR', SNR_DB, 'dB')

## 4. 載入 40 案例（中文 20 + 台語 20）

音檔以 `case_id.wav` 解析；若 bundle 只放了 10 條範例，會自動只跑存在的音檔。

In [ ]:
import pandas as pd

def collect(csv_path, audio_dir, language):
    df = pd.read_csv(csv_path).fillna('')
    cases = []
    for _, r in df.iterrows():
        wav = audio_dir / f"{r['case_id']}.wav"
        if wav.exists():
            cases.append(dict(case_id=r['case_id'], language=language,
                              category=r.get('category', ''),
                              ref=r['reference_text'], wav=wav))
    return cases

cases = collect(MANDARIN_CSV, MANDARIN_AUDIO, 'mandarin') + \
        collect(HOKKIEN_CSV,  HOKKIEN_AUDIO,  'hokkien')
print(f'cases with audio present: {len(cases)}  '
      f"(mandarin={sum(c['language']=='mandarin' for c in cases)}, "
      f"hokkien={sum(c['language']=='hokkien' for c in cases)})")

## 5. 推論：每條音檔 × 4 噪音

支援斷點續跑（每 10 列存檔一次）。

In [ ]:
import time
from tqdm.auto import tqdm

if OUT_CSV.exists():
    rows = pd.read_csv(OUT_CSV).fillna('').to_dict('records')
    done = {(r['case_id'], r['noise']) for r in rows}
    print(f'resume: {len(done)} runs already done')
else:
    rows, done = [], set()

t0 = time.time()
for c in tqdm(cases, desc='cases'):
    for nt in NOISE_TYPES:
        if (c['case_id'], nt) in done:
            continue
        try:
            audio, sr = load_noisy(c['wav'], nt)
            hyp = transcribe(audio, sr)
            cer = cer_percent(hyp, c['ref'])
            err = ''
        except Exception as e:
            hyp, cer, err = '', float('nan'), f'{type(e).__name__}: {e}'
        rows.append(dict(case_id=c['case_id'], language=c['language'],
                         category=c['category'], noise=nt,
                         dur_sec=round(len(librosa.load(str(c['wav']), sr=16000)[0])/16000, 2),
                         cer=round(cer, 2) if cer == cer else cer,
                         hyp=hyp, ref=c['ref'], error=err))
        if len(rows) % 10 == 0:
            pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print(f'done {len(rows)} runs in {time.time()-t0:.0f}s → {OUT_CSV}')

## 6. 彙整 CER（語言 × 噪音）

In [ ]:
df = pd.read_csv(OUT_CSV)
piv = df.pivot_table(index='language', columns='noise', values='cer', aggfunc='mean').round(2)
piv = piv[[n for n in NOISE_TYPES if n in piv.columns]]
print('平均 CER (%)：')
print(piv)
print('\n整體平均 CER: {:.2f}%'.format(df['cer'].mean()))

## 7. 下載結果 CSV（可選）

In [ ]:
from google.colab import files
files.download(str(OUT_CSV))